# 📘 Regime Detector Refactor Pipeline

## 🎯 Goal

Simplify regime detection into 3 independent regime components:

1. Volatility Regime
2. Trend Regime
3. Volume Regime

Each component:
- computes its own metrics,
- assigns its own label,
- returns its own notes/warnings.

Final regime label is only a concatenation of the 3 component labels.

---

# ✅ Proposed Pipeline

```text
processed_df
    ↓
prepare returns + sorted dataframe
    ↓
detect_volatility_regime()
    ↓
detect_trend_regime()
    ↓
detect_volume_regime()
    ↓
build_final_regime_label()
    ↓
compute_regime_confidence()
    ↓
return regime_report
```

---

# 1️⃣ Volatility Regime

## Metrics

```text
current_vol = std(last 20 daily returns)
rolling_vol = rolling 20-day std of daily returns
vol_percentile = percentile rank of current_vol inside rolling_vol
```

---

## Mathematical Formulation

20-day volatility:

\[
\sigma_{20,current} = std(r_{t-19}, ..., r_t)
\]

Rolling historical volatility:

\[
\sigma_{20,historical}
\]

Percentile rank:

\[
Percentile =
\frac{
\#(\sigma_{20,historical} \le \sigma_{20,current})
}{
N
}
\]

---

## Labels

| Condition | Label |
|---|---|
| `vol_percentile < 0.25` | `LOW_VOLATILITY` |
| `0.25 <= vol_percentile < 0.75` | `NORMAL_VOLATILITY` |
| `0.75 <= vol_percentile < 0.92` | `HIGH_VOLATILITY` |
| `vol_percentile >= 0.92` | `EXTREME_VOLATILITY` |

---

## Meaning

| Label | Interpretation |
|---|---|
| `LOW_VOLATILITY` | Market is calmer than usual |
| `NORMAL_VOLATILITY` | Typical volatility regime |
| `HIGH_VOLATILITY` | Elevated uncertainty |
| `EXTREME_VOLATILITY` | Panic/speculative regime |

---

# 2️⃣ Trend Regime

## Metrics

```text
ma_gap = (MA7 - MA21) / MA21
return_20d = close.pct_change(20)
return_5d = close.pct_change(5)
```

---

## Mathematical Formulation

Moving average gap:

\[
MA\_gap =
\frac{
MA_7 - MA_{21}
}{
MA_{21}
}
\]

20-day return:

\[
R_{20} =
\frac{
Close_t - Close_{t-20}
}{
Close_{t-20}
}
\]

5-day return:

\[
R_5 =
\frac{
Close_t - Close_{t-5}
}{
Close_{t-5}
}
\]

---

## Labels

Keep trend labels simple:

| Condition | Label |
|---|---|
| `ma_gap > 0.015 and return_20d > 0.03` | `UPTREND` |
| `ma_gap < -0.015 and return_20d < -0.03` | `DOWNTREND` |
| `abs(ma_gap) <= 0.015 and abs(return_20d) <= 0.04` | `SIDEWAYS` |
| otherwise | `MIXED_TREND` |

---

## Meaning

| Label | Interpretation |
|---|---|
| `UPTREND` | Medium-term bullish structure |
| `DOWNTREND` | Medium-term bearish structure |
| `SIDEWAYS` | No strong directional trend |
| `MIXED_TREND` | Signals conflict or transition regime |

---

## Important Refactor

### ❌ Remove

```text
REVERSAL_RISK as fallback label
```

Reason:
- Too broad,
- difficult to explain,
- mixes uncertainty with actual reversal behavior.

---

## Use `return_5d` only for warnings

### Warnings

| Condition | Warning |
|---|---|
| `trend_regime == UPTREND and return_5d < -0.02` | `SHORT_TERM_PULLBACK` |
| `trend_regime == DOWNTREND and return_5d > 0.02` | `SHORT_TERM_REBOUND` |

---

## Meaning

| Warning | Interpretation |
|---|---|
| `SHORT_TERM_PULLBACK` | Temporary weakness inside uptrend |
| `SHORT_TERM_REBOUND` | Temporary rebound inside downtrend |

---

# 3️⃣ Volume Regime

## Important Rename

Rename:

```text
liquidity_regime
```

to:

```text
volume_regime
```

Reason:
- clearer,
- easier to explain,
- directly tied to volume metrics.

---

## Metrics

```text
volume_zscore =
(latest_volume - mean(last 60 volume))
/
std(last 60 volume)

recent_volume_ratio =
mean(last 5 volume)
/
mean(last 60 volume)
```

---

## Mathematical Formulation

Volume z-score:

\[
Z_{volume} =
\frac{
Volume_t - \mu_{volume,60}
}{
\sigma_{volume,60}
}
\]

Recent volume ratio:

\[
VolumeRatio =
\frac{
mean(Volume_{t-4}, ..., Volume_t)
}{
mean(Volume_{t-59}, ..., Volume_t)
}
\]

---

## Labels

| Condition | Label |
|---|---|
| `volume_zscore >= 2.5 or recent_volume_ratio >= 1.8` | `VOLUME_SPIKE` |
| `recent_volume_ratio <= 0.45` | `LOW_VOLUME` |
| otherwise | `NORMAL_VOLUME` |

---

## Meaning

| Label | Interpretation |
|---|---|
| `VOLUME_SPIKE` | Abnormally high market participation |
| `LOW_VOLUME` | Weak liquidity / low participation |
| `NORMAL_VOLUME` | Typical trading activity |

---

# 4️⃣ Final Regime Label

Final label should ONLY concatenate the 3 regime labels.

---

## Construction

```python
final_regime_label = (
    f"{volatility_regime}__{trend_regime}__{volume_regime}"
)
```

---

## Examples

```text
HIGH_VOLATILITY__UPTREND__VOLUME_SPIKE

NORMAL_VOLATILITY__SIDEWAYS__NORMAL_VOLUME

EXTREME_VOLATILITY__DOWNTREND__LOW_VOLUME
```

---

# 5️⃣ Recommended Output Structure

```python
regime_report = {
    "volatility_regime": volatility_regime,
    "trend_regime": trend_regime,
    "volume_regime": volume_regime,

    "final_regime_label": final_regime_label,

    "metrics": {
        "vol_percentile": vol_percentile,
        "current_vol_20d": current_vol,

        "ma_gap": ma_gap,
        "return_20d": return_20d,
        "return_5d": return_5d,

        "volume_zscore": volume_zscore,
        "recent_volume_ratio": recent_volume_ratio,
    },

    "warnings": warnings,

    "regime_confidence": confidence,

    "regime_notes": notes,
}
```

---

# 6️⃣ What To Remove

## Remove or avoid:

```text
REVERSAL_RISK as fallback label
market_regime
risk_label
stable/risky/unstable combined labels
complex interpretation layer
```

---

## Reason

The combination:

```text
volatility_regime
+
trend_regime
+
volume_regime
```

already fully describes market state.

Additional meta-labels:
- increase complexity,
- create ambiguity,
- and make agents harder to debug.

---

# 7️⃣ Compatibility Note

For backward compatibility:

temporarily keep:

```python
"liquidity_regime": volume_regime
```

But internally prefer:

```python
"volume_regime"
```

This prevents immediate graph/node breakage during refactor.

---

# 8️⃣ Final Refactor Principle

```text
Do not make RegimeDetector a decision engine.
```

Regime detector should ONLY describe market state:

```text
Volatility state
+
Trend state
+
Volume state
```

NOT:
- retraining decision,
- governance decision,
- recommendation logic,
- risk management logic.

Those should belong to separate downstream agents/nodes.

---

# ✅ Final Simplified Architecture

```text
processed_df
    ↓
Volatility Regime
    ↓
Trend Regime
    ↓
Volume Regime
    ↓
Final Regime Label
    ↓
Return Structured Regime Report
```

This keeps:
- logic modular,
- labels explainable,
- state compact,
- and LangGraph reasoning much cleaner.